# Aprendizado Nao Supervisionado — Observabilidade de Microsservicos

**Objetivo:** Descobrir padroes nos dados de observabilidade do spring-petclinic-cloud  
coletados em 540 experimentos (3 topologias x 6 RPS x 30 repeticoes).

**Dataset:** `dataset_final.csv` — 29.015 linhas x 36 colunas  
**Fonte:** Prometheus + Zipkin + Loki

---
## Pipeline
1. Instalacao e imports
2. Carregamento
3. Agregacao (1 linha por minuto de experimento)
4. Selecao de features
5. Engenharia de features
6. Normalizacao
7. Correlacao
8. PCA
9. K-Means
10. DBSCAN
11. t-SNE / UMAP
12. Validacao (ARI)
13. Exportacao

In [ ]:
!pip install -q umap-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
import umap

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11, 'figure.dpi': 120})
SEED = 42
np.random.seed(SEED)
print('Imports OK')

## 2. Carregamento
> **Opcao A:** upload manual | **Opcao B:** Google Drive

In [ ]:
# --- Opcao A: Upload direto ---
from google.colab import files
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]

# --- Opcao B: Google Drive (descomente) ---
# from google.colab import drive
# drive.mount('/content/drive')
# CSV_PATH = '/content/drive/MyDrive/TCC/dataset_final.csv'

df_raw = pd.read_csv(CSV_PATH, parse_dates=['timestamp_min'])
print(f'Shape bruto: {df_raw.shape}')
display(df_raw.head(3))

In [ ]:
print('=== Distribuicao das topologias ===')
print(df_raw['network_topology'].value_counts())
print('\n=== Cargas (RPS) ===', sorted(df_raw['load_rps'].unique()))
print('\n=== Servicos ===', df_raw['service'].unique())
null_pct = (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
nulos = null_pct[null_pct > 0]
print('\n=== Nulos por coluna ===')
print(nulos if len(nulos) else 'Nenhum nulo!')

## 3. Agregacao — Uma linha por minuto de experimento

O dataset bruto tem multiplas linhas por minuto (uma por servico x node_ip).  
Agregamos para **(topologia, carga, minuto)** = uma linha por minuto de experimento.

**Estrategia:**
- `max` entre node_ips por servico → elimina duplicacao de pods
- `mean` entre os 4 servicos → agrega a visao do cluster

In [ ]:
META_COLS = ['timestamp_min', 'service', 'node_ip', 'network_topology', 'cluster_nodes', 'load_rps']
NUMERIC_COLS = [c for c in df_raw.columns if c not in META_COLS]
print(f'Features numericas ({len(NUMERIC_COLS)}): {NUMERIC_COLS}')

# Passo 1: max por (topologia, carga, minuto, servico) — elimina duplicacao de node_ip
step1 = (
    df_raw
    .groupby(['network_topology', 'load_rps', 'timestamp_min', 'service'])[NUMERIC_COLS]
    .max().reset_index()
)

# Passo 2: mean por (topologia, carga, minuto) — agrega entre servicos
df_agg = (
    step1
    .groupby(['network_topology', 'load_rps', 'timestamp_min'])[NUMERIC_COLS]
    .mean().reset_index()
)

print(f'\nShape apos agregacao: {df_agg.shape}')
print(df_agg.groupby('network_topology').size().rename('linhas'))
display(df_agg.head(3))

## 4. Selecao de features

Removemos redundancias para evitar multicolinearidade:

In [ ]:
REMOVE = [
    'mrt_ms',                  # alias de avg_total_duration_ms
    'net_rx_bytes',            # versao nao normalizada de net_rx_bps
    'net_tx_bytes',            # versao nao normalizada de net_tx_bps
    'memory_heap_used_bytes',  # capturado por memory_utilization_pct
    'memory_heap_max_bytes',   # capturado por memory_utilization_pct
    'db_bottleneck_ratio',     # derivada de avg_db / avg_total
    'error_rate',              # sempre 0 — sem variancia
]

FEATURE_COLS = [c for c in NUMERIC_COLS if c not in REMOVE]
print(f'Features selecionadas ({len(FEATURE_COLS)}):')
for f in FEATURE_COLS:
    print(f'  * {f}')

meta = df_agg[['network_topology', 'load_rps', 'timestamp_min']].copy()
X_raw = df_agg[FEATURE_COLS].copy()
print(f'\nX_raw shape: {X_raw.shape}')

## 5. Engenharia de features

In [ ]:
X_eng = X_raw.copy()

# Razao DB / Total de latencia
X_eng['db_ratio'] = (
    X_eng['avg_db_total_duration_ms'] /
    X_eng['avg_total_duration_ms'].replace(0, np.nan)
).fillna(0).clip(0, 1)

# Pressao no pool de conexoes
total_conn = X_eng['db_connections_active'] + X_eng['db_connections_idle']
X_eng['connection_pressure'] = (
    X_eng['db_connections_active'] / total_conn.replace(0, np.nan)
).fillna(0)

# Trafego total de rede
X_eng['net_total_bps'] = X_eng['net_rx_bps'] + X_eng['net_tx_bps']

print(f'Features apos engenharia: {X_eng.shape[1]}')
display(X_eng.describe().round(4))

## 6. Normalizacao

In [ ]:
# Remove features com variancia zero
zero_var = X_eng.columns[X_eng.var() == 0].tolist()
if zero_var:
    print(f'Removendo variancia zero: {zero_var}')
    X_eng = X_eng.drop(columns=zero_var)

# Clip outliers (p1 - p99)
for col in X_eng.columns:
    lo, hi = X_eng[col].quantile([0.01, 0.99])
    X_eng[col] = X_eng[col].clip(lo, hi)

# StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_eng)
X_scaled_df = pd.DataFrame(X_scaled, columns=X_eng.columns, index=X_eng.index)

print(f'X_scaled shape: {X_scaled_df.shape}')
print(f'Media maxima absoluta: {X_scaled_df.mean().abs().max():.2e}  (esperado ~0)')
print(f'Std media: {X_scaled_df.std().mean():.4f}  (esperado ~1)')

## 7. Correlacao entre features

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
corr = X_eng.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            vmin=-1, vmax=1, annot=False, linewidths=0.3, ax=ax)
ax.set_title('Correlacao entre Features', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## 8. PCA — Reducao de Dimensionalidade

In [ ]:
pca_full = PCA(random_state=SEED).fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = int(np.argmax(cumvar >= 0.95)) + 1
n_90 = int(np.argmax(cumvar >= 0.90)) + 1

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(range(1, len(cumvar)+1), pca_full.explained_variance_ratio_*100, color='#2196F3', alpha=0.8)
axes[0].set(xlabel='Componente', ylabel='Variancia Explicada (%)', title='Por Componente')

axes[1].plot(range(1, len(cumvar)+1), cumvar*100, 'o-', color='#FF5722', lw=2)
axes[1].axhline(90, color='green',  ls='--', label=f'90% -> {n_90} PCs')
axes[1].axhline(95, color='orange', ls='--', label=f'95% -> {n_95} PCs')
axes[1].set(xlabel='N componentes', ylabel='Variancia Acumulada (%)', title='Acumulada')
axes[1].legend()

plt.suptitle('Analise PCA', fontweight='bold')
plt.tight_layout()
plt.show()
print(f'PCs para 90%: {n_90} | para 95%: {n_95}')

In [ ]:
TOPO_COLORS = {'single_node': '#2196F3', 'same_network': '#4CAF50', 'distributed': '#FF5722'}
loads_sorted = sorted(meta['load_rps'].unique())
cmap_rps = plt.cm.get_cmap('plasma', len(loads_sorted))

# PCA 2D para visualizacao
pca2 = PCA(n_components=2, random_state=SEED)
X_pca2 = pca2.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for topo, color in TOPO_COLORS.items():
    mask = meta['network_topology'] == topo
    axes[0].scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=color, label=topo, alpha=0.6, s=30, edgecolors='none')
axes[0].set(xlabel=f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)',
            ylabel=f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)',
            title='PCA 2D — Topologia')
axes[0].legend(); axes[0].spines[['top','right']].set_visible(False)

for i, rps in enumerate(loads_sorted):
    mask = meta['load_rps'] == rps
    axes[1].scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=[cmap_rps(i)], label=f'{rps} RPS', alpha=0.6, s=30, edgecolors='none')
axes[1].set(xlabel=f'PC1', ylabel=f'PC2', title='PCA 2D — Carga (RPS)')
axes[1].legend(title='Carga', fontsize=8); axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('PCA 2D', fontweight='bold'); plt.tight_layout(); plt.show()

In [ ]:
# Loadings: quais features mais explicam cada componente?
loadings = pd.DataFrame(pca2.components_.T, index=X_eng.columns, columns=['PC1','PC2']).abs()
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for i, pc in enumerate(['PC1','PC2']):
    top = loadings[pc].sort_values(ascending=False).head(10)
    top.plot.barh(ax=axes[i], color=['#2196F3','#FF5722'][i], alpha=0.8)
    axes[i].set(title=f'Top 10 features — {pc}', xlabel='Loading |valor|')
    axes[i].invert_yaxis(); axes[i].spines[['top','right']].set_visible(False)
plt.suptitle('Contribuicao das Features nos Componentes Principais', fontweight='bold')
plt.tight_layout(); plt.show()

## 9. K-Means

In [ ]:
# PCA com n_95 componentes para clustering
pca_n = PCA(n_components=n_95, random_state=SEED)
X_pca_n = pca_n.fit_transform(X_scaled)
print(f'Usando {n_95} PCs ({pca_n.explained_variance_ratio_.sum()*100:.1f}% variancia)')

K_range = range(2, 11)
inertias, silhouettes = [], []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    lbl = km.fit_predict(X_pca_n)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_pca_n, lbl))

best_k = list(K_range)[np.argmax(silhouettes)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_range), inertias, 'o-', color='#2196F3', lw=2)
axes[0].set(xlabel='K', ylabel='Inertia (WCSS)', title='Metodo do Cotovelo')
axes[0].spines[['top','right']].set_visible(False)

axes[1].plot(list(K_range), silhouettes, 'o-', color='#FF5722', lw=2)
axes[1].axvline(best_k, color='green', ls='--', label=f'K={best_k} (melhor)')
axes[1].set(xlabel='K', ylabel='Silhouette Score', title='Score de Silhouette')
axes[1].legend(); axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('K otimo para K-Means', fontweight='bold')
plt.tight_layout(); plt.show()
print(f'K com melhor Silhouette: {best_k} (score={max(silhouettes):.4f})')

In [ ]:
K_BEST = best_k  # ajuste manualmente se quiser testar outro K
km_final = KMeans(n_clusters=K_BEST, random_state=SEED, n_init=10)
meta = meta.copy()
meta['kmeans_label'] = km_final.fit_predict(X_pca_n)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
cmap_k = plt.cm.get_cmap('tab10', K_BEST)
for k in range(K_BEST):
    mask = meta['kmeans_label'] == k
    axes[0].scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=[cmap_k(k)], label=f'Cluster {k}', alpha=0.6, s=30, edgecolors='none')
axes[0].set(title=f'K-Means (K={K_BEST})', xlabel='PC1', ylabel='PC2')
axes[0].legend(); axes[0].spines[['top','right']].set_visible(False)

for topo, color in TOPO_COLORS.items():
    mask = meta['network_topology'] == topo
    axes[1].scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=color, label=topo, alpha=0.6, s=30, edgecolors='none')
axes[1].set(title='Topologia Real (referencia)', xlabel='PC1', ylabel='PC2')
axes[1].legend(); axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('K-Means vs Topologia Real', fontweight='bold')
plt.tight_layout(); plt.show()

print('\n=== Clusters vs Topologia ===')
display(pd.crosstab(meta['kmeans_label'], meta['network_topology']))
print('\n=== Clusters vs Carga ===')
display(pd.crosstab(meta['kmeans_label'], meta['load_rps']))

## 10. DBSCAN

In [ ]:
nbrs = NearestNeighbors(n_neighbors=5).fit(X_pca_n)
distances, _ = nbrs.kneighbors(X_pca_n)
k_dist = np.sort(distances[:, -1])

plt.figure(figsize=(9, 4))
plt.plot(k_dist, color='#2196F3', lw=1.5)
plt.xlabel('Ponto (ordenado)'); plt.ylabel('Distancia ao 5o vizinho (eps)')
plt.title('k-distancia — escolha eps no cotovelo')
plt.grid(alpha=0.4); plt.tight_layout(); plt.show()

In [ ]:
EPS = 1.5        # ajuste conforme o grafico k-distancia
MIN_SAMPLES = 5

db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
meta['dbscan_label'] = db.fit_predict(X_pca_n)
n_clusters = len(set(meta['dbscan_label'])) - (1 if -1 in meta['dbscan_label'].values else 0)
n_noise    = (meta['dbscan_label'] == -1).sum()
print(f'Clusters: {n_clusters} | Anomalias: {n_noise} ({n_noise/len(meta)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(9, 5.5))
for lbl in sorted(meta['dbscan_label'].unique()):
    mask  = meta['dbscan_label'] == lbl
    color = 'black' if lbl == -1 else plt.cm.tab10(lbl)
    name  = 'Anomalia' if lbl == -1 else f'Cluster {lbl}'
    ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=[color],
               label=name, alpha=0.7 if lbl != -1 else 0.3, s=25 if lbl != -1 else 12, edgecolors='none')
ax.set(title=f'DBSCAN (eps={EPS}, min_samples={MIN_SAMPLES})', xlabel='PC1', ylabel='PC2')
ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

print('\n=== DBSCAN vs Topologia ===')
display(pd.crosstab(meta['dbscan_label'], meta['network_topology']))

## 11. t-SNE e UMAP

In [ ]:
print('Calculando t-SNE...')
X_tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000).fit_transform(X_scaled)
print('Calculando UMAP...')
X_umap = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=15, min_dist=0.1).fit_transform(X_scaled)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for row, (X_embed, name) in enumerate([(X_tsne, 't-SNE'), (X_umap, 'UMAP')]):
    for topo, color in TOPO_COLORS.items():
        mask = meta['network_topology'] == topo
        axes[row, 0].scatter(X_embed[mask, 0], X_embed[mask, 1], c=color, label=topo, alpha=0.6, s=25, edgecolors='none')
    axes[row, 0].set_title(f'{name} — Topologia', fontweight='bold')
    axes[row, 0].legend(fontsize=9); axes[row, 0].spines[['top','right']].set_visible(False)

    for i, rps in enumerate(loads_sorted):
        mask = meta['load_rps'] == rps
        axes[row, 1].scatter(X_embed[mask, 0], X_embed[mask, 1], c=[cmap_rps(i)], label=f'{rps} RPS', alpha=0.6, s=25, edgecolors='none')
    axes[row, 1].set_title(f'{name} — Carga (RPS)', fontweight='bold')
    axes[row, 1].legend(title='Carga', fontsize=8); axes[row, 1].spines[['top','right']].set_visible(False)

plt.suptitle('Visualizacao t-SNE e UMAP', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

## 12. Validacao — Adjusted Rand Index (ARI)

Mede o quanto os clusters encontrados coincidem com os rotulos reais (topologia).  
**1.0** = perfeito | **0.0** = aleatorio

In [ ]:
le = LabelEncoder()
topo_encoded = le.fit_transform(meta['network_topology'])

ari_km = adjusted_rand_score(topo_encoded, meta['kmeans_label'])
mask_valid = meta['dbscan_label'] != -1
ari_db = adjusted_rand_score(topo_encoded[mask_valid], meta['dbscan_label'][mask_valid])

print('=== Adjusted Rand Index vs topologia real ===')
print(f'  K-Means (K={K_BEST}): ARI = {ari_km:.4f}')
print(f'  DBSCAN:               ARI = {ari_db:.4f}')
print()
print('ARI ~1.0: clusters correspondem as topologias (modelo descobriu sem supervisao)')
print('ARI ~0.0: clusters nao tem relacao com as topologias (padroes diferentes encontrados)')

## 13. Exportacao

In [ ]:
df_export = pd.concat([
    meta.reset_index(drop=True),
    X_scaled_df.reset_index(drop=True)
], axis=1)

EXPORT_PATH = 'dataset_preprocessado.csv'
df_export.to_csv(EXPORT_PATH, index=False)
print(f'Exportado: {EXPORT_PATH} — shape: {df_export.shape}')
display(df_export.head(3))

from google.colab import files
files.download(EXPORT_PATH)